# 실습 6 – PreLLM

# 실습의 개념적 흐름

- 이전 실습에서 배운 Transformer
- Pre-LLM 연결 단계
- Hugging Face를 이용한 실제 사전 학습된 LLM
- LoRA를 이용한 LLM 적응
- RAG를 이용한 LLM에 외부 지식 제공

# Pre-LLM 실습:

## 우리가 만든 Transformer에서 LLM으로

# 전후 비교

| 단계 | 실습에서 만든 Transformer | 현대 LLM |
|---|---|---|
| 1. 입력 | 텍스트 프롬프트, 예: `"i want"` | 텍스트 프롬프트, 예: `"I want"` |
| 2. 토큰화 | `prompt.lower().split()` | LLM 토크나이저(BPE 등) |
| 3. 토큰 ID | `word_to_id` | `tokenizer.encode()` |
| 4. 임베딩 | 토큰 임베딩 + 위치 임베딩 | 토큰 임베딩 + 위치 정보 |
| 5. Attention | Causal Multi-Head Self-Attention | Causal Multi-Head Attention(아키텍처에 따라 다른 변형) |
| 6. FFN | Linear → ReLU → Linear | 일반적으로 게이트 FFN 사용, 예: SwiGLU |
| 7. Transformer 블록 | 블록 1개 | 여러 블록을 쌓은 구조 |
| 8. 출력 | `logits = model(x)` | `logits = model(input_ids)` |
| 9. 다음 토큰 점수 | `logits[0, -1]` | 마지막 위치의 logits |
| 10. 토큰 선택 | Temperature + Top-k Sampling | Greedy, Sampling 등 |
| 11. 추가 | `ids.append(next_id)` | 생성된 토큰 ID 추가 |
| 12. 반복 | `ids[-MAX_LEN:]`을 다시 처리 | 일반적으로 KV 캐시를 사용하여 반복 |
| 13. 출력 텍스트 | `id_to_word` | `tokenizer.decode()` |

# 전후 비교

## 가장 중요한 차이점

- **토큰화:** 우리가 만든 모델은 단어 수준의 ID를 사용한다. 현대 LLM은 일반적으로 서브워드 토큰을 사용한다.
- **모델 아키텍처:** 우리가 구현한 모델은 Transformer 블록 하나를 사용하지만, LLM은 추가적인 아키텍처 개선 사항이 적용된 여러 블록을 쌓는다.
- **텍스트 생성:** 기본 원리는 동일하다. Logits를 계산하고, 다음 토큰을 선택하고, 해당 ID를 추가하는 과정을 반복한다.
- **추론 효율성:** 우리가 구현한 모델은 매 단계마다 문맥을 다시 계산한다. 현대 LLM은 일반적으로 KV 캐시를 사용해 이전 Attention 계산의 상당 부분을 반복하지 않는다.

# 공통 설정 – 패키지 설치

- 추가 패키지가 필요하다.

```bash
python -m pip install -U transformers accelerate peft bitsandbytes scikit-learn
```

- **transformers:** NLP와 멀티모달 AI를 위한 표준 Hugging Face 라이브러리이다. GPT, Llama, Qwen과 같은 LLM을 다운로드하고 불러와 사용하는 데 필수적이다.
- **accelerate:** 다중 GPU 환경 또는 혼합 정밀도(FP16/BF16) 환경에서 PyTorch 모델을 쉽게 실행할 수 있도록 하는 Hugging Face 유틸리티이다.
- **peft (Parameter-Efficient Fine-Tuning):** LLM을 파인튜닝하기 위한 LoRA 등의 기법을 제공한다.
- **bitsandbytes:** 모델 가중치를 8비트 또는 4비트 정밀도로 양자화하는 데 사용된다. 이를 통해 GPU 메모리 사용량을 크게 줄여, 일반 소비자용 하드웨어에서도 대규모 모델을 실행하거나 파인튜닝할 수 있다.
- **scikit-learn:** 전통적인 머신러닝 알고리즘(분류, 회귀, 클러스터링 등), 데이터 전처리, 성능 평가 지표를 제공한다.

# 공통 설정 – 모델 다운로드

- 1.5B 모델은 RTX 4060 Ti(8GB VRAM)에 적합하다.
  - 모델 가중치: 1.5B × 2바이트 = 3GB (16FP)
  - 실행 시: KV 캐시 + 활성화 값
  - 파인튜닝 시: 파인튜닝 스크립트, 기울기, 옵티마이저 상태

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

- **AutoTokenizer:** 특정 모델에 맞는 올바른 토크나이저를 자동으로 선택하고 불러온다. 사람의 텍스트를 토큰 ID로 변환하거나 그 반대로 변환한다.
- **AutoModelForCausalLM:** "Causal Language Modeling"을 뜻한다. 다음 토큰 예측과 같은 텍스트 생성용으로 설계된 올바른 모델 아키텍처를 자동으로 불러온다.

# BPE / 서브워드 토큰화 1

- Hugging Face 토크나이저로 시작한다.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

text = "He wanted to eat Chinese food and behaveed uncharacteristically."

tokens = tokenizer.tokenize(text)

ids = tokenizer.encode(text)

print(tokens)
print(ids)

이 코드는 이전 실습의 `sentence.split()`에 해당한다.

이전 실습: 단어 → 단어 ID  
LLM: 텍스트 → 서브워드 토큰 → 토큰 ID

`Ġ`는 앞에 공백이 있음을 나타내는 표기이다. 따라서 `"want"`와 `"Ġwant"`는 서로 다른 토큰 ID를 가진다.

# BPE / 서브워드 토큰화 2

In [ ]:
for token_id in ids:
    print(
        token_id,
        tokenizer.decode([token_id])
    )

(Qwen 2.5, 15억 개 파라미터 버전)

각 모델은 일반적으로 자체 어휘에 맞게 특별히 설계된 고유한 토크나이저를 가지고 있다.

**토큰화:** 모델의 어휘를 기준으로 텍스트를 더 작은 조각(토큰)으로 나눈다. 토큰은 전체 단어, 서브워드 또는 개별 문자 및 문장 부호일 수 있다.

**토큰 ID:** 각 텍스트 토큰을 고유한 정수 ID로 변환한다.

단어 수준: `"unbelievable"`  
↓  
`["unbelievable"]`

서브워드/토큰 수준: `"unbelievable"`  
↓  
`["un", "believ", "able"]`

# 입력 텐서

In [ ]:
inputs = tokenizer(
    text,
    return_tensors="pt"
)

print(inputs["input_ids"])
print(inputs["input_ids"].shape)

print(inputs["attention_mask"])
print(inputs["attention_mask"].shape)

print(inputs)

- **`tokenizer(text, ...)`:** 텍스트를 한 번에 처리하는 방법이다. 토큰화, ID 변환, 필요한 특수 토큰(Attention Mask 또는 Padding 등) 추가를 자동으로 처리한다.
- **`return_tensors="pt"`:** `pt`는 PyTorch를 뜻한다. 기본적으로 토크나이저는 숫자로 이루어진 Python 리스트(예: `[40, 937, 311...]`)를 반환한다. 이 옵션은 해당 숫자를 PyTorch 텐서로 변환한다.
- **`inputs`:** 토큰 ID와 Attention Mask가 포함될 수 있는 딕셔너리와 유사한 객체이다.
- **`input_ids`:** 토큰을 나타내는 정수 ID이다. (배치 1개와 토큰 7개의 경우 Shape는 `[1, 7]`이다.)
- **`attention_mask`:** 1 = 유효한 토큰, 0 = Padding 토큰.

# 실제 Causal LM 불러오기

다음 토큰 생성을 위한 모델 클래스:

$$
L = -\sum_t \log P(x_t \mid x_1, \ldots, x_{t-1})
$$

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
)
model.eval()

- 실제 신경망의 가중치를 다운로드하고 불러온다.
- 기본 32비트 대신 16비트 부동소수점 정밀도(Half Precision)를 사용하여 모델을 불러온다.
- 사용 가능한 하드웨어(GPU 또는 CPU)를 자동으로 감지하고 모델 레이어를 장치에 효율적으로 배치한다. 단일 GPU가 있으면 전체 모델을 GPU VRAM에 직접 불러온다.

# Qwen 1.5B 모델 구조

<p align="center"><img src="images/img-2026-11-18-11-15-45.png" width="80%"/></p>

# Qwen 아키텍처 살펴보기

In [ ]:
print(model.config)

| 특징 | 실습 Transformer | Qwen2.5 1.5B | 현대 LLM |
|---|---|---|---|
| 어휘 크기 | 약 2천 토큰 | 151,936 | 수만에서 수십만 |
| 임베딩 차원 | 64 | 1,536 | 수천 |
| Attention Head | 4 | 12(Q), 2(KV) | 약 128개 이상 |
| Transformer 레이어 | 1 | 28 | 수십 개 이상 |
| 모델 파라미터 | 작음(약 10만) | 1.54B | 수십억 개 이상 |
| 학습 토큰 | 수천 개 | ?? | 수십억에서 수조 개 |

# 실제 Causal LM 불러오기

## 입력을 장치로 이동

In [ ]:
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

## Logits 확인

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

print("input_ids:", inputs["input_ids"].shape)
print("logits:", logits.shape)

Logits의 Shape: `(batch_size, seq_len, vocab_size)`

# 실제 Causal LM 불러오기

## 다음 토큰의 Logits 추출

In [ ]:
next_logits= logits[0,-1]
print(next_logits.shape)

## Logits를 확률로 변환하고 Top-k 계산

In [ ]:
probs = torch.softmax(next_logits, dim=-1)
top_probs, top_ids = torch.topk(probs, k=10)
for prob, token_id in zip(top_probs, top_ids):
    token = tokenizer.decode([token_id.item()])
    print(repr(token), f"{prob.item():4f}")

`repr(token)`은 문자열을 Python의 표현 형식으로 감싼다. 이를 통해 앞쪽 공백, 탭, 줄바꿈 표시(예: `\n`)처럼 눈에 보이지 않는 서식 문자를 명확하게 확인할 수 있다.

## Manually Generate One Token

LLMs generate text autoregressively - meaning they output one token at a time.

<p align="center"><img src="images/img-2026-11-18-11-39-20.png" width="30%"/></p>

In [ ]:
next_id = top_ids[0]
new_ids = torch.cat(
    [inputs[“input_ids”], next_id.view(1,1)], dim=-1
)
print(tokenizer.decode(new_ids[0]))

next_id.view(1, 1)
- Takes the newly predicted token ID (next_id) and reshapes it into a 2D tensor
of shape [1, 1] (Batch size = 1, Sequence length = 1). Reshaping is necessary
so its dimensions match inputs["input_ids"] before they can be joined.

torch.cat([..., ...], dim=1)
- Concatenation along the sequence dimension (dim=1 / columns).